In [ ]:
from qiskit import __version__

print(__version__)

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

In [ ]:
c = "1"
d = "0"


In [ ]:
protocol = QuantumCircuit(2)

# prepare ebit used for superdense coding.
# Alice とBobの間にベル状態の一つを作る。
protocol.h(0)
protocol.cx(0, 1) # 制御Qubit、目的Qubit

# Alice's operations
# 4種類のベル状態に転移する。
if d == "1":
    protocol.z(0)
if c == "1":
    protocol.x(0)
protocol.barrier()

# Bob's actions
protocol.cx(0, 1)
protocol.h(0)
protocol.measure_all()

display(protocol.draw(output="mpl"))

In [ ]:
result = AerSimulator().run(protocol).result()
statistics = result.get_counts()

for outcome, frequency in statistics.items():
    print(f"Measured {outcome} with frequency {frequency}")

display(plot_histogram(statistics))

In [ ]:
# コイン乱数を生成するためのQubit
rbg = QuantumRegister(1, "coin")
# AliceのQubit
ebit0 = QuantumRegister(1, "A")
# BobのQubit
ebit1 = QuantumRegister(1, "B")

# Aliceの古典ビット
Alice_c = ClassicalRegister(1, "Alice c")
Alice_d = ClassicalRegister(1, "Alice_d")

test = QuantumCircuit(rgb, ebit0, ebit1, Alice_d, Alice_c)

# 2つのebitをベル状態に
test.h(ebit0)
test.cx(ebit0, ebit1)
test.barrier()

# 二つの乱数をAliceの古典ビットに入れる
test.h(rbg)
test.measure(rgb, Alice_c)
test.h(rgb)
test.measure(rgb, Alice_d)
test.barrier()

# Aliceの古典ビットの状態により、4種のベル状態生成
with test.if_test((Alice_d, 1), label="Z"):
    test.z(ebit0)
with test.if_test((Alice_c, 1), label="X"):
    test.x(ebit0)
test.barrier()

# Bobのアクション
test.cx(ebit0, ebit1)
test.h(ebit0)
test.barrier()

# Bobの古典ビットを接続
Bob_c = ClassicalRegister(1, "Bob c")
Bob_d = ClassicalRegister(1, "Bob d")
test.add_register(Bob_d)
test.add_register(Bob_c)

test.measure(ebit0, Bob_d)
test.measure(ebit1, Bob_c)

display(test.draw(output="mpl"))


In [ ]:
result = AerSimulator().run(test).result()
statistics = result.get_counts()
display(plot_histogram(statistics))